In [ ]:
import torch

def predict(text, tokenizer, embedding, lstm, classifier, device="cpu"):
    embedding.eval()
    lstm.eval()
    classifier.eval()

    with torch.no_grad():
        # 1. text -> list of token ids
        token_ids = tokenizer.encode(text)              # list[int]

        # 2. list -> tensor + batch dimension
        batch_x = torch.tensor(token_ids, dtype=torch.long)
        batch_x = batch_x.unsqueeze(0)                   # [1, seq_len]
        batch_x = batch_x.to(device)

        # 3. forward pass (same as evaluation)
        embedded_x = embedding(batch_x)
        out, (h_n, _) = lstm(embedded_x)
        final_repr = h_n[0]
        logits = classifier(final_repr)

        # 4. sigmoid + threshold
        prob = torch.sigmoid(logits).item()
        pred = 1 if prob >= 0.5 else 0

    return pred, prob
